# 4 — Analysis & Visualisation

Generate publication-quality figures:
- Training curves (loss & accuracy)
- Comparison bar charts
- Attention heatmaps overlaid on images
- Per-token text attention bar charts
- Qualitative prediction grids

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import json
from pathlib import Path

RESULTS = Path("../results")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 4.1 — Training curves

In [ ]:
from visualization.plot_results import load_history, plot_training_curves

histories = {
    "Asymmetric": load_history(RESULTS / "metrics" / "asymmetric_s42_history.json"),
    "Symmetric":  load_history(RESULTS / "metrics" / "symmetric_s42_history.json"),
}
fig = plot_training_curves(histories, save_path=FIGURES / "training_curves.png")
fig.show()

## 4.2 — Comparison bar chart

In [ ]:
from visualization.plot_results import plot_comparison_bar
from training.evaluate import evaluate_from_checkpoint

comparison = {}
for name, ckpt_name in [("Asymmetric", "asymmetric_s42"), ("Symmetric", "symmetric_s42")]:
    ckpt = RESULTS / "checkpoints" / f"{ckpt_name}_best.pt"
    comparison[name] = evaluate_from_checkpoint(ckpt, data_dir="../data")

fig = plot_comparison_bar(comparison, save_path=FIGURES / "comparison_bar.png")
fig.show()

## 4.3 — Attention heatmaps

In [ ]:
from models import AsymmetricVQAModel
from data.preprocess import VQADataset, build_answer_vocab, get_image_transform
from visualization.attention_maps import (
    get_attention_weights, plot_image_attention, plot_text_attention, decode_tokens,
)

# Load best asymmetric model
ckpt = torch.load(RESULTS / "checkpoints" / "asymmetric_s42_best.pt", map_location=device, weights_only=False)
model = AsymmetricVQAModel(num_answers=1000).to(device)
model.load_state_dict(ckpt["model_state_dict"])

# Load a small validation set
answer_to_idx, idx_to_answer = build_answer_vocab(
    "../data/answers/v2_mscoco_train2014_annotations.json", top_k=1000,
)
val_ds = VQADataset(
    questions_file="../data/questions/v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file="../data/answers/v2_mscoco_val2014_annotations.json",
    image_dir="../data/images/val2014",
    answer_to_idx=answer_to_idx,
    transform=get_image_transform("val"),
    max_samples=100,
)

In [ ]:
# Pick a sample and visualise
for i in range(min(5, len(val_ds))):
    image, input_ids, attention_mask, answer_idx = val_ds[i]
    question = val_ds.samples[i]["question"]
    tokens = decode_tokens(input_ids)

    attn = get_attention_weights(model, image, input_ids, attention_mask, device)

    fig = plot_image_attention(image, attn["txt_to_img"], question, tokens)
    fig.savefig(FIGURES / f"attn_img_{i}.png", dpi=150, bbox_inches="tight")

    fig = plot_text_attention(attn["img_to_txt"], tokens, question)
    fig.savefig(FIGURES / f"attn_txt_{i}.png", dpi=150, bbox_inches="tight")

## 4.4 — Qualitative comparison grid

In [ ]:
from models import SymmetricVQAModel
from visualization.qualitative_examples import qualitative_grid

# Load symmetric model too
ckpt_sym = torch.load(RESULTS / "checkpoints" / "symmetric_s42_best.pt", map_location=device, weights_only=False)
model_sym = SymmetricVQAModel(num_answers=1000).to(device)
model_sym.load_state_dict(ckpt_sym["model_state_dict"])

models_dict = {"Asymmetric": model, "Symmetric": model_sym}

fig = qualitative_grid(
    models_dict,
    val_ds,
    idx_to_answer,
    n_samples=6,
    device=device,
    save_path=str(FIGURES / "qualitative_grid.png"),
)
fig.show()